# Vigil — Notebook 03: Clasificación Semántica NLP con Gemini

Este cuaderno implementa el etiquetado semántico del discurso político (sentimiento, framing y palabras clave) utilizando la API de Gemini 2.5-flash.

In [ ]:
import polars as pl
import os
import sys
from dotenv import load_dotenv
sys.path.append('..')
from src.analysis.semantic_agent import AgenteSemanticoElectoral

# Cargar clave de API (.env)
load_dotenv("../.env")
print("GEMINI_API_KEY configurada:", "GEMINI_API_KEY" in os.environ)

## 1. Cargar Datos Limpios de la Capa Silver

In [ ]:
posts = pl.read_parquet("../data/silver/redes_sociales/fb_posts_clean.parquet")
comments = pl.read_parquet("../data/silver/redes_sociales/fb_comments_clean.parquet")

# MVP: Asignamos el candidato correspondiente a cada activo
posts = posts.with_columns(pl.lit("Beatriz Estrada").alias("candidato"))
comments = comments.with_columns(pl.lit("Geraldine Ponce").alias("candidato"))

print("Posts para análisis NLP:", posts.height)
print("Comentarios para análisis NLP:", comments.height)

## 2. Inicializar Agente Semántico y Ejecutar Inferencia Estructurada

In [ ]:
agente = AgenteSemanticoElectoral()

# Correr inferencia (si no hay API key, utilizará el fallback de simulación)
# Para optimizar el tiempo de MVP, limitamos la inferencia de prueba si es necesario
# pero aquí procesaremos todo el conjunto (60 posts, 110 comentarios)
posts_enriched = agente.clasificar_dataframe(posts, "texto_publicacion")
comments_enriched = agente.clasificar_dataframe(comments, "texto_publicacion")

print("Clasificación NLP finalizada.")

## 3. Muestra de Resultados del Análisis Semántico

In [ ]:
print("Resultados de Posts (Framing & Sentiment):")
print(posts_enriched.select(["candidato", "sentimiento", "framing", "keywords_extracted"]).head(5))

print("\nResultados de Comentarios (Framing & Sentiment):")
print(comments_enriched.select(["candidato", "sentimiento", "framing", "keywords_extracted"]).head(5))

## 4. Persistir Resultados en Capa Gold

In [ ]:
os.makedirs("../data/gold/redes_sociales", exist_ok=True)

posts_enriched.write_parquet("../data/gold/redes_sociales/fb_posts_gold.parquet")
comments_enriched.write_parquet("../data/gold/redes_sociales/fb_comments_gold.parquet")

print("Fase 3 Completada: Datos consolidados guardados en data/gold/")